# BirdCLEF+ 2026

This notebook is my first implementation of a solution for the BirdCLEF competition. Nothing much, just trying to get a working notebook first. With keras, I'll use Efficient Net V2 on log-mel spectrograms for my initial baseline.

Architecture : EfficientNetV2-B2 on log-mel spectrograms

Framework    : TensorFlow / Keras


NOTEBOOK STRUCTURE

0.  Install & Imports
1.  Configuration (all hyperparams in one place)
2.  Data Loading & Label Encoding
3.  Soundscape Labels → Training Rows
4.  Cross-Validation Split
5.  Audio Preprocessing (waveform → mel spectrogram)
6.  Augmentation (SpecAugment, MixUp, pink noise)
7.  tf.data Pipeline
8.  Model Definition (EfficientNetV2-B2 + custom head)
9.  Focal Loss & LR Schedule
10.  Training Loop
11.  Soundscape Inference (sliding window)
12.  Submission Builder

# INSTALL AND IMPORTS

In [1]:
# Uncomment on first run:
!pip install -q librosa colorednoise

In [2]:
import os
import gc
import math
import random
import warnings
from pathlib import Path

In [3]:
warnings.filterwarnings("ignore")

In [4]:
import numpy as np
import pandas as pd
import librosa
import colorednoise as cn
from sklearn.model_selection import StratifiedKFold

In [5]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import mixed_precision 

2026-04-14 15:18:12.827181: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776179893.077410      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776179893.151258      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776179893.763748      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776179893.763810      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776179893.763813      55 computation_placer.cc:177] computation placer alr

In [6]:
print(f"TensorFlow  : {tf.__version__}")
print(f"GPUs found  : {tf.config.list_physical_devices('GPU')}")

TensorFlow  : 2.19.0
GPUs found  : []


2026-04-14 15:18:39.901825: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


# 1. CONFIGURATION

In [7]:
class CFG:
    BASE_DIR               = Path("/kaggle/input/competitions/birdclef-2026/")
    TRAIN_AUDIO_DIR        = BASE_DIR / "train_audio"
    TRAIN_SOUNDSCAPES_DIR  = BASE_DIR / "train_soundscapes"
    TEST_SOUNDSCAPES_DIR   = BASE_DIR / "test_soundscapes"
    TRAIN_CSV              = BASE_DIR / "train.csv"
    TAXONOMY_CSV           = BASE_DIR / "taxonomy.csv"
    SOUNDSCAPE_LABELS_CSV  = BASE_DIR / "train_soundscapes_labels.csv"
    SAMPLE_SUB_CSV         = BASE_DIR / "sample_submission.csv"

    
    SAMPLE_RATE   = 32_000
    DURATION      = 5             
    N_SAMPLES     = SAMPLE_RATE * DURATION  

    # Mel spectrogram 
    N_FFT       = 1024
    HOP_LENGTH  = 320    # 32000/320 = 100 frames/sec → 500 frames for 5 s
    N_MELS      = 128
    FMIN        = 20     # Hz
    FMAX        = 16000  # Hz — 32 kHz Nyquist is 16 kHz

    #  Image fed into the CNN
    IMG_HEIGHT  = 128
    IMG_WIDTH   = 384    # we resize the ~500-frame spectrogram to this
    IMG_SHAPE   = (IMG_HEIGHT, IMG_WIDTH, 3)

    #Model
    BACKBONE    = "EfficientNetV2B2"
    DROP_RATE   = 0.3
    DROP_RATE2  = 0.2

    # Training
    BATCH_SIZE      = 32
    EPOCHS          = 3
    LEARNING_RATE   = 1e-3
    MIN_LR          = 1e-6
    WEIGHT_DECAY    = 1e-4
    WARMUP_EPOCHS   = 3
    PATIENCE        = 7            # early stopping
    N_FOLDS         = 5
    TRAIN_FOLDS     = [0]          # fold(s) to train; add more for CV ensemble
    SEED            = 42

    # Augmentation
    USE_MIXUP          = True
    MIXUP_ALPHA        = 0.4
    USE_SPEC_AUGMENT   = True
    FREQ_MASK_MAX      = 20        # max frequency bins to mask
    TIME_MASK_MAX      = 40        # max time steps to mask
    USE_NOISE          = True
    NOISE_PROB         = 0.5
    SNR_DB_MIN         = 5
    SNR_DB_MAX         = 30

    # Label smoothing
    # Secondary labels get this weight (< 1.0 = uncertain)
    SECONDARY_LABEL_WEIGHT = 0.5

    # Quality filter
    # XC ratings 1–5; 0 = no rating (iNat). Use 0 to keep everything.
    MIN_RATING = 0.0

    #  Mixed precision
    USE_MIXED_PRECISION = True

    #  Inference
    SOUNDSCAPE_DURATION  = 60     # seconds (test files are 1 min)
    INFER_OVERLAP        = 0.0    # fraction — 0 = non-overlapping 5-s chunks
    INFER_BATCH_SIZE     = 64

In [8]:
def set_seed(seed: int = CFG.SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

In [9]:
set_seed()

In [10]:
if CFG.USE_MIXED_PRECISION:
    mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision: ON (float16 compute, float32 variables)")

Mixed precision: ON (float16 compute, float32 variables)


# 2. DATA LOADING & LABEL ENCODING

In [11]:
train_df   = pd.read_csv(CFG.TRAIN_CSV)
taxonomy   = pd.read_csv(CFG.TAXONOMY_CSV)
sample_sub = pd.read_csv(CFG.SAMPLE_SUB_CSV)
sc_labels  = pd.read_csv(CFG.SOUNDSCAPE_LABELS_CSV)

In [12]:
# Submission column order defines our class order — never deviate from this
ALL_SPECIES   = [c for c in sample_sub.columns if c != "row_id"]
SPECIES_TO_IDX = {s: i for i, s in enumerate(ALL_SPECIES)}
NUM_CLASSES    = len(ALL_SPECIES)

In [13]:
print(f"\nTotal species (classes)  : {NUM_CLASSES}")
print(f"Total training files     : {len(train_df)}")


Total species (classes)  : 234
Total training files     : 35549


In [14]:
# Build full filepath 
train_df["filepath"] = train_df["filename"].apply(
    lambda x: str(CFG.TRAIN_AUDIO_DIR / x)
)

In [15]:
#  Quality filter 
if CFG.MIN_RATING > 0:
    train_df = train_df[
        (train_df["rating"] >= CFG.MIN_RATING) | (train_df["rating"] == 0)
    ].reset_index(drop=True)

In [16]:
# Drop species not in the submission
train_df = train_df[
    train_df["primary_label"].isin(SPECIES_TO_IDX)
].reset_index(drop=True)

train_df["label_idx"] = train_df["primary_label"].map(SPECIES_TO_IDX)

In [17]:
print(f"After quality filter     : {len(train_df)} samples")
print(f"\nMost common species (top 5):")
print(train_df["primary_label"].value_counts().head())
print(f"\nLeast common species (bottom 5):")
print(train_df["primary_label"].value_counts().tail())

After quality filter     : 35549 samples

Most common species (top 5):
primary_label
rubthr1    499
banana     498
fepowl     497
soulap1    497
houspa     496
Name: count, dtype: int64

Least common species (bottom 5):
primary_label
209233    2
516975    1
23724     1
116570    1
23150     1
Name: count, dtype: int64


# 3. SOUNDSCAPE LABELS → TRAINING ROWS

In [18]:
# These are expert-annotated 5-second segments from field recordings —
# the SAME domain as the test set. They're extremely valuable.

def timestamp_to_seconds(ts) -> float:
    """
    Convert a timestamp to seconds.
    Handles both numeric seconds (e.g. 15.0) and HH:MM:SS / MM:SS strings.
    """
    try:
        return float(ts)
    except (ValueError, TypeError):
        pass
    # String timestamp: HH:MM:SS or MM:SS
    parts = str(ts).strip().split(":")
    parts = [float(p) for p in parts]
    if len(parts) == 3:
        return parts[0] * 3600 + parts[1] * 60 + parts[2]
    elif len(parts) == 2:
        return parts[0] * 60 + parts[1]
    else:
        return parts[0]
 

In [19]:
 
def parse_soundscape_labels(labels_df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert annotated soundscape segments into training rows compatible
    with the main train_df schema.
    """
    rows = []
    for _, row in labels_df.iterrows():
        fp = str(CFG.TRAIN_SOUNDSCAPES_DIR / row["filename"])
        if not os.path.exists(fp):
            continue
 
        raw_labels = str(row["primary_label"]).split(";")
        valid_labels = [l.strip() for l in raw_labels if l.strip() in SPECIES_TO_IDX]
        if not valid_labels:
            continue
 
        rows.append({
            "filepath"          : fp,
            "primary_label"     : valid_labels[0],
            "secondary_labels"  : ",".join(valid_labels[1:]),
            "label_idx"         : SPECIES_TO_IDX[valid_labels[0]],
            "start_sec"         : timestamp_to_seconds(row["start"]),
            "end_sec"           : timestamp_to_seconds(row["end"]),
            "rating"            : 5.0,   # treat expert labels as high quality
            "is_soundscape"     : True,
        })
 
    return pd.DataFrame(rows)
 
 
sc_train_df = parse_soundscape_labels(sc_labels)
print(f"\nSoundscape training segments : {len(sc_train_df)}")
if len(sc_train_df) > 0:
    sc_species = sc_train_df["primary_label"].nunique()
    print(f"Unique species in soundscape labels : {sc_species}")


Soundscape training segments : 1478
Unique species in soundscape labels : 29


# 4. CROSS-VALIDATION SPLIT

In [20]:
skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
train_df["fold"] = -1

for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df["label_idx"])):
    train_df.loc[val_idx, "fold"] = fold

print(f"\nFold distribution:")
print(train_df["fold"].value_counts().sort_index())


Fold distribution:
fold
0    7110
1    7110
2    7110
3    7110
4    7109
Name: count, dtype: int64


# 5. AUDIO PREPROCESSING

In [21]:
def _make_mel_filterbank() -> tf.Tensor:
    """
    Returns a (n_fft/2 + 1, N_MELS) float32 matrix mapping
    linear STFT bins to mel bins.
    Uses TF's built-in linear_to_mel_weight_matrix.
    """
    n_stft_bins = CFG.N_FFT // 2 + 1
    return tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins           = CFG.N_MELS,
        num_spectrogram_bins   = n_stft_bins,
        sample_rate            = CFG.SAMPLE_RATE,
        lower_edge_hertz       = CFG.FMIN,
        upper_edge_hertz       = CFG.FMAX,
        dtype                  = tf.float32,
    )  # (n_stft_bins, N_MELS)

In [39]:
MEL_FILTERBANK = _make_mel_filterbank()

In [22]:
def load_audio(
    filepath: str,
    sr: int = CFG.SAMPLE_RATE,
    offset: float = 0.0,
    duration: float = None,
) -> np.ndarray:
    """Load .ogg with librosa; returns float32 waveform or silence."""
    try:
        audio, _ = librosa.load(filepath, sr=sr, offset=offset,
                                duration=duration, mono=True)
        return audio.astype(np.float32)
    except Exception:
        return np.zeros(CFG.N_SAMPLES, dtype=np.float32)

In [23]:
def load_and_pad_cpu(
    filepath: str,
    start_sec: float = -1.0,
    training: bool   = False,
) -> np.ndarray:
    """
    CPU-side only: load OGG + optional noise + crop/pad.
    Returns a flat float32 waveform of length N_SAMPLES.
    """
    offset = float(start_sec) if start_sec >= 0 else 0.0
    audio  = load_audio(filepath, offset=offset,
                        duration=CFG.DURATION if start_sec >= 0 else None)

    # Pink noise augmentation (CPU, uses numpy)
    if training and CFG.USE_NOISE and np.random.random() < CFG.NOISE_PROB:
        noise = cn.powerlaw_psd_gaussian(1, len(audio)).astype(np.float32)
        snr   = np.random.uniform(CFG.SNR_DB_MIN, CFG.SNR_DB_MAX)
        sig_p   = np.mean(audio ** 2) + 1e-9
        noi_p   = np.mean(noise ** 2) + 1e-9
        noise  *= np.sqrt((sig_p / (10 ** (snr / 10.0))) / noi_p)
        audio  += noise

    # Crop / repeat-pad to exactly N_SAMPLES
    n = CFG.N_SAMPLES
    if len(audio) == 0:
        return np.zeros(n, dtype=np.float32)
    if len(audio) >= n:
        start = np.random.randint(0, len(audio) - n + 1)
        return audio[start : start + n]
    repeats = math.ceil(n / len(audio))
    return np.tile(audio, repeats)[:n]

In [24]:
@tf.function(jit_compile=False)
def waveform_to_image(waveform: tf.Tensor) -> tf.Tensor:
    """
    Full GPU pipeline: float32 waveform (N_SAMPLES,) →
    float32 image (IMG_HEIGHT, IMG_WIDTH, 3).

    Steps:
      1. STFT        — short-time Fourier transform
      2. Power       — magnitude² of complex STFT
      3. Mel         — multiply by filterbank matrix
      4. Log         — compress dynamic range (≈ dB)
      5. Normalise   — per-sample min-max → [0, 1]
      6. Resize      — to IMG_HEIGHT × IMG_WIDTH
      7. 3-channel   — replicate to pseudo-RGB
    """
    stft = tf.signal.stft(
        waveform,
        frame_length = CFG.N_FFT,
        frame_step   = CFG.HOP_LENGTH,
        fft_length   = CFG.N_FFT,
        window_fn    = tf.signal.hann_window,
        pad_end      = True,
    )

    power = tf.math.real(stft * tf.math.conj(stft))  # |stft|²

    mel = tf.tensordot(power, MEL_FILTERBANK, axes=[[1], [0]])
    mel.set_shape([None, CFG.N_MELS])

    log_mel = tf.math.log(mel + 1e-6)                # (time, n_mels)
    
    mn  = tf.reduce_min(log_mel)
    mx  = tf.reduce_max(log_mel)
    log_mel = (log_mel - mn) / (mx - mn + 1e-8)

    log_mel = tf.transpose(log_mel)                   # (n_mels, time)

    img = tf.expand_dims(log_mel, axis=-1)            # (H, W, 1)
    img = tf.image.resize(img, [CFG.IMG_HEIGHT, CFG.IMG_WIDTH])

    img = tf.concat([img, img, img], axis=-1)         # (H, W, 3)
    return img

In [25]:
def process_sample(
    filepath: str,
    start_sec: float = -1.0,
    training: bool   = False,
) -> np.ndarray:
    """
    CPU path only — returns raw waveform as numpy.
    The GPU path (waveform_to_image) is called inside the tf.data map.
    Used only for inference outside the tf.data pipeline.
    """
    audio = load_and_pad_cpu(filepath, start_sec=start_sec, training=training)
    img   = waveform_to_image(tf.constant(audio))
    return img.numpy()

# 6. AUGMENTATION  (SpecAugment + MixUp)

In [26]:
@tf.function
def spec_augment(image: tf.Tensor) -> tf.Tensor:
    """
    SpecAugment: randomly mask contiguous frequency bands and time steps.
    Applied per-image inside the dataset pipeline.
    Reference: Park et al. 2019 (https://arxiv.org/abs/1904.08779)
    """
    H, W, C = CFG.IMG_HEIGHT, CFG.IMG_WIDTH, 3

    # ── Frequency masking ──────────────────────────────────────
    f = tf.random.uniform((), 0, CFG.FREQ_MASK_MAX, dtype=tf.int32)
    f0 = tf.random.uniform((), 0, H - f, dtype=tf.int32)
    # Build a mask: 1 everywhere except the masked band
    freq_mask = tf.concat([
        tf.ones([f0,     W, C], tf.float32),
        tf.zeros([f,     W, C], tf.float32),
        tf.ones([H-f0-f, W, C], tf.float32),
    ], axis=0)
    image = image * freq_mask

    # ── Time masking ───────────────────────────────────────────
    t = tf.random.uniform((), 0, CFG.TIME_MASK_MAX, dtype=tf.int32)
    t0 = tf.random.uniform((), 0, W - t, dtype=tf.int32)
    time_mask = tf.concat([
        tf.ones([H, t0,     C], tf.float32),
        tf.zeros([H, t,     C], tf.float32),
        tf.ones([H, W-t0-t, C], tf.float32),
    ], axis=1)
    image = image * time_mask

    return image

In [27]:
@tf.function
def apply_mixup(images: tf.Tensor, labels: tf.Tensor) -> tuple:
    """
    MixUp: convex combination of two random training examples.
    Reference: Zhang et al. 2018 (https://arxiv.org/abs/1710.09412)

    Operates on a full batch — call AFTER batching in the pipeline.
    """
    B = tf.shape(images)[0]

    # Sample lambda from Beta(alpha, alpha); enforce lam >= 0.5
    lam = tf.random.stateless_uniform(
        [B], seed=[CFG.SEED, 0], minval=0.0, maxval=1.0
    )
    lam = tf.maximum(lam, 1.0 - lam)

    # Shuffle indices for the second sample
    idx = tf.random.shuffle(tf.range(B))
    img2 = tf.gather(images, idx)
    lbl2 = tf.gather(labels, idx)

    lam_img = tf.reshape(tf.cast(lam, tf.float32), [B, 1, 1, 1])
    lam_lbl = tf.reshape(tf.cast(lam, tf.float32), [B, 1])

    mixed_images = lam_img * images + (1.0 - lam_img) * img2
    mixed_labels = lam_lbl * labels + (1.0 - lam_lbl) * lbl2

    return mixed_images, mixed_labels

# 7. TF.DATA PIPELINE

In [28]:
def make_target_vector(
    primary_label: str,
    secondary_labels_str: str = "",
) -> np.ndarray:
    """
    Multi-hot float32 target vector of length NUM_CLASSES.
    Primary label  → 1.0
    Secondary labels → SECONDARY_LABEL_WEIGHT (soft target; uncertain)
    """
    target = np.zeros(NUM_CLASSES, dtype=np.float32)

    if primary_label in SPECIES_TO_IDX:
        target[SPECIES_TO_IDX[primary_label]] = 1.0

    if secondary_labels_str:
        for s in str(secondary_labels_str).split(","):
            s = s.strip().strip("[]'\"\\ ")
            if s in SPECIES_TO_IDX:
                idx = SPECIES_TO_IDX[s]
                if target[idx] == 0.0:
                    target[idx] = CFG.SECONDARY_LABEL_WEIGHT

    return target

In [29]:
def _py_load_waveform(filepath_b, primary_b, secondary_b, start_sec_f, training_b):
    """
    CPU-only py_function: loads OGG → waveform + target vector.
    Returns (waveform float32 [N_SAMPLES], target float32 [NUM_CLASSES])
    """
    filepath  = filepath_b.numpy().decode("utf-8")
    primary   = primary_b.numpy().decode("utf-8")
    secondary = secondary_b.numpy().decode("utf-8")
    start_sec = float(start_sec_f.numpy())
    training  = bool(training_b.numpy())

    waveform = load_and_pad_cpu(filepath, start_sec=start_sec, training=training)
    target   = make_target_vector(primary, secondary)
    return waveform, target

In [30]:
@tf.function
def _gpu_process(waveform: tf.Tensor, target: tf.Tensor, training: bool) -> tuple:
    """
    GPU-side tf.function:
      waveform → mel image, optional SpecAugment
    """
    img = waveform_to_image(waveform)
    if training and CFG.USE_SPEC_AUGMENT:
        img = spec_augment(img)
    return img, target

In [31]:
def build_dataset(df: pd.DataFrame, training: bool = False) -> tf.data.Dataset:
    filepaths   = df["filepath"].astype(str).values
    primaries   = df["primary_label"].astype(str).values
    secondaries = df.get("secondary_labels",
                         pd.Series([""] * len(df))).fillna("").astype(str).values
    start_secs  = df.get("start_sec",
                         pd.Series([-1.0] * len(df))).fillna(-1.0).astype(np.float32).values
    training_flag = np.array([training] * len(df), dtype=bool)

    # ── Stage 1: CPU load ───────────────────────────────────────
    def tf_load_waveform(fp, prim, sec, start, train_f):
        waveform, target = tf.py_function(
            func  = _py_load_waveform,
            inp   = [fp, prim, sec, start, train_f],
            Tout  = [tf.float32, tf.float32],
        )
        waveform.set_shape((CFG.N_SAMPLES,))
        target.set_shape((NUM_CLASSES,))
        return waveform, target

    # ── Stage 2: GPU process ────────────────────────────────────
    def tf_process(waveform, target):
        return _gpu_process(waveform, target, training)

    ds = tf.data.Dataset.from_tensor_slices((
        filepaths, primaries, secondaries, start_secs, training_flag
    ))

    if training:
        ds = ds.shuffle(buffer_size=min(len(df), 8192), seed=CFG.SEED,
                        reshuffle_each_iteration=True)

    # Stage 1: parallel CPU loading (fill the waveform buffer)
    ds = ds.map(tf_load_waveform, num_parallel_calls=tf.data.AUTOTUNE)

    # Batch BEFORE stage 2 so GPU processes whole batches at once
    ds = ds.batch(CFG.BATCH_SIZE, drop_remainder=training)

    # Stage 2: GPU spectrogram + augmentation (per-batch)
    ds = ds.map(
        lambda wav_batch, tgt_batch: tf.vectorized_map(
            lambda pair: tf_process(pair[0], pair[1]),
            (wav_batch, tgt_batch)
        ),
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    if training and CFG.USE_MIXUP:
        ds = ds.map(apply_mixup, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

# 8. MODEL

In [32]:
def build_model(num_classes: int = NUM_CLASSES) -> keras.Model:
    """
    EfficientNetV2-B2 + custom classification head.

    Key choices:
    • ImageNet pre-training gives meaningful low-level feature detectors
      (edges, textures) that transfer well to spectrograms.
    • GlobalAveragePooling over the spatial feature map.
    • Two-stage dropout head with BatchNorm for training stability.
    • Sigmoid output (multi-label) with float32 dtype even under mixed
      precision (avoids numerical issues with loss computation).
    """
    inputs = keras.Input(shape=CFG.IMG_SHAPE, name="spectrogram")

    # ── Backbone ────────────────────────────────────────────────
    backbone = tf.keras.applications.EfficientNetV2B2(
        include_top=False,
        weights="imagenet",
        input_shape=CFG.IMG_SHAPE,
    )
    # Fine-tune the whole network (recommended for audio domain shift)
    backbone.trainable = True

    x = backbone(inputs, training=True)

    # ── Head ────────────────────────────────────────────────────
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(CFG.DROP_RATE, name="drop1")(x)
    x = layers.Dense(512, name="fc1")(x)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.Activation("swish", name="swish1")(x)
    x = layers.Dropout(CFG.DROP_RATE2, name="drop2")(x)

    # float32 output — important for numerical stability with focal loss
    outputs = layers.Dense(
        num_classes,
        activation="sigmoid",
        dtype="float32",
        name="logits",
    )(x)

    model = keras.Model(inputs, outputs, name="BirdCLEF_EfficientNetV2B2")
    return model

# 9. FOCAL LOSS & LR SCHEDULE

In [34]:
def focal_loss(gamma: float = 2.0, alpha: float = 0.25):
    """
    Binary focal loss for multi-label classification.

    Focal loss down-weights easy negatives, focusing training on the
    hard/rare examples — critical when most of the 234 classes are
    absent in any given 5-second window.

    Reference: Lin et al. 2017 (https://arxiv.org/abs/1708.02002)
    """
    def _loss(y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)

        eps    = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)

        bce   = -(y_true * tf.math.log(y_pred) +
                  (1 - y_true) * tf.math.log(1 - y_pred))
        p_t   = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        a_t   = y_true * alpha  + (1 - y_true) * (1 - alpha)
        focal = a_t * tf.pow(1.0 - p_t, gamma) * bce

        return tf.reduce_mean(focal)

    return _loss

In [35]:
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    """
    Linear warm-up for `warmup_steps`, then cosine decay to `min_lr`.

    Warm-up avoids destroying ImageNet weights in the first few batches
    when the head (randomly initialised) produces large gradients.
    """

    def __init__(
        self,
        base_lr: float,
        min_lr: float,
        total_steps: int,
        warmup_steps: int,
    ):
        super().__init__()
        self.base_lr      = float(base_lr)
        self.min_lr       = float(min_lr)
        self.total_steps  = float(total_steps)
        self.warmup_steps = float(warmup_steps)

    def __call__(self, step: tf.Tensor) -> tf.Tensor:
        step = tf.cast(step, tf.float32)
        # Linear warm-up
        warmup_lr = self.base_lr * step / self.warmup_steps
        # Cosine decay
        progress  = (step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        cosine_lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (
            1.0 + tf.cos(math.pi * progress)
        )
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return dict(
            base_lr=self.base_lr,
            min_lr=self.min_lr,
            total_steps=self.total_steps,
            warmup_steps=self.warmup_steps,
        )

# 10. TRAINING LOOP

In [36]:
def train_fold(fold: int) -> tuple:
    print(f"\n{'═'*60}")
    print(f"  FOLD {fold}")
    print(f"{'═'*60}")

    # ── Build DataFrames ────────────────────────────────────────
    fold_train = train_df[train_df["fold"] != fold].copy()
    fold_val   = train_df[train_df["fold"] == fold].copy()

    # Add annotated soundscape segments to training (not validation)
    if len(sc_train_df) > 0:
        sc_copy = sc_train_df.copy()
        sc_copy["fold"] = -1
        sc_copy["secondary_labels"] = sc_copy.get("secondary_labels",
                                                   pd.Series([""] * len(sc_copy)))
        fold_train = pd.concat([fold_train, sc_copy], ignore_index=True)

    print(f"  Train rows : {len(fold_train)}")
    print(f"  Val rows   : {len(fold_val)}")

    # ── Build tf.data datasets ──────────────────────────────────
    train_ds = build_dataset(fold_train, training=True)
    val_ds   = build_dataset(fold_val,   training=False)

    # ── LR schedule ─────────────────────────────────────────────
    steps_per_epoch = max(1, len(fold_train) // CFG.BATCH_SIZE)
    total_steps     = steps_per_epoch * CFG.EPOCHS
    warmup_steps    = steps_per_epoch * CFG.WARMUP_EPOCHS

    lr_schedule = WarmupCosineDecay(
        base_lr      = CFG.LEARNING_RATE,
        min_lr       = CFG.MIN_LR,
        total_steps  = total_steps,
        warmup_steps = warmup_steps,
    )

    # ── Model ───────────────────────────────────────────────────
    model = build_model()
    model.compile(
        optimizer = keras.optimizers.AdamW(
            learning_rate = lr_schedule,
            weight_decay  = CFG.WEIGHT_DECAY,
        ),
        loss    = focal_loss(gamma=2.0, alpha=0.25),
        metrics = [
            # Multi-label ROC-AUC matches the competition metric
            keras.metrics.AUC(
                curve      = "ROC",
                multi_label= True,
                name       = "roc_auc",
            ),
        ],
    )

    if fold == CFG.TRAIN_FOLDS[0]:
        model.summary(line_length=80)

    # ── Callbacks ───────────────────────────────────────────────
    ckpt_path = f"model_fold{fold}.keras"
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath        = ckpt_path,
            monitor         = "val_roc_auc",
            mode            = "max",
            save_best_only  = True,
            verbose         = 1,
        ),
        keras.callbacks.EarlyStopping(
            monitor              = "val_roc_auc",
            mode                 = "max",
            patience             = CFG.PATIENCE,
            restore_best_weights = True,
            verbose              = 1,
        ),
        keras.callbacks.CSVLogger(f"history_fold{fold}.csv"),
    ]

    # ── Train ───────────────────────────────────────────────────
    history = model.fit(
        train_ds,
        validation_data = val_ds,
        epochs          = CFG.EPOCHS,
        callbacks       = callbacks,
        verbose         = 1,
    )

    best_auc = max(history.history.get("val_roc_auc", [0.0]))
    print(f"\n  ✓ Fold {fold} best val ROC-AUC : {best_auc:.4f}")

    return model, history


In [37]:
trained_models = []
fold_histories = []

In [40]:
for fold in CFG.TRAIN_FOLDS:
    model, history = train_fold(fold)
    trained_models.append(model)
    fold_histories.append(history)
    gc.collect()

print("\nTraining complete.")


════════════════════════════════════════════════════════════
  FOLD 0
════════════════════════════════════════════════════════════
  Train rows : 29917
  Val rows   : 7110
35839040/35839040 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "BirdCLEF_EfficientNetV2B2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                      ┃ Output Shape             ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ spectrogram (InputLayer)          │ (None, 128, 384, 3)      │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ efficientnetv2-b2 (Functional)    │ (None, 4, 12, 1408)      │     8,769,374 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)      │ (None, 1408)             │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ drop1 (Dropout)                   │ (None, 1408)             │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ fc1 (Dense)                       │ (None, 512)              │       721,408 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)          │ (None, 512)              │         2,048 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ swish1 (Activation)               │ (None, 512)              │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ drop2 (Dropout)                   │ (None, 512)              │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ logits (Dense)                    │ (None, 234)              │       120,042 │
└───────────────────────────────────┴──────────────────────────┴───────────────┘

 Total params: 9,612,872 (36.67 MB)

 Trainable params: 9,529,560 (36.35 MB)

 Non-trainable params: 83,312 (325.44 KB)

Epoch 1/3


InvalidArgumentError: Graph execution error:

Detected at node ones_3/pfor/TensorArrayV2Stack/TensorListStack defined at (most recent call last):
<stack traces unavailable>
Error in user-defined function passed to ParallelMapDatasetV2:8 transformation with iterator: Iterator::Root::Prefetch::ParallelMapV2::ParallelMapV2: PartialTensorShape: Incompatible shapes during merge: [128,13,3] vs. [128,210,3]
	 [[{{node ones_3/pfor/TensorArrayV2Stack/TensorListStack}}]]
	 [[IteratorGetNext]] [Op:__inference_multi_step_on_iterator_99271]

# 11. SOUNDSCAPE INFERENCE — SLIDING WINDOW
Test soundscapes are 60-second files.
We split them into 5-second non-overlapping chunks (12 per file)
and predict species presence for each chunk.

row_id format: <stem>_<end_second>
e.g.  BC2026_Test_0001_S05_20250227_010002_20
                                             
                                             ^^ end = 20 s → window 15–20 s

In [ ]:
def load_soundscape_chunks(
    filepath: str,
    duration: float  = CFG.SOUNDSCAPE_DURATION,
    chunk_dur: float = float(CFG.DURATION),
    overlap: float   = CFG.INFER_OVERLAP,
) -> list:
    """
    Load a full soundscape and return (end_second, waveform) pairs.
    Spectrogram conversion happens later in a batched GPU call.
    """
    try:
        audio, _ = librosa.load(filepath, sr=CFG.SAMPLE_RATE,
                                duration=duration, mono=True)
        audio = audio.astype(np.float32)
    except Exception:
        audio = np.zeros(int(duration * CFG.SAMPLE_RATE), dtype=np.float32)

    step   = chunk_dur * (1.0 - overlap)
    chunks = []
    start  = 0.0

    while round(start + chunk_dur, 6) <= duration + 1e-6:
        s     = int(start * CFG.SAMPLE_RATE)
        e     = s + CFG.N_SAMPLES
        chunk = audio[s:e]
        # Pad last chunk if needed
        if len(chunk) < CFG.N_SAMPLES:
            chunk = np.pad(chunk, (0, CFG.N_SAMPLES - len(chunk)))
        end_sec = int(round(start + chunk_dur))
        chunks.append((end_sec, chunk))
        start += step

    return chunks

In [ ]:
def predict_soundscape(filepath: str, models: list) -> list:
    """
    Run inference on one soundscape file.
    Waveforms → images on GPU in one batched call.
    """
    filename = Path(filepath).stem
    chunks   = load_soundscape_chunks(filepath)

    waveforms = np.stack([w for _, w in chunks], axis=0)   # (N, N_SAMPLES)
    end_secs  = [e for e, _ in chunks]

    # GPU: batch waveform → images
    images = tf.vectorized_map(waveform_to_image,
                               tf.constant(waveforms))     # (N, H, W, 3)

    all_model_preds = []
    for m in models:
        preds = []
        for i in range(0, len(images), CFG.INFER_BATCH_SIZE):
            batch = images[i : i + CFG.INFER_BATCH_SIZE]
            p = m.predict(batch, verbose=0)
            preds.append(p)
        all_model_preds.append(np.concatenate(preds, axis=0))

    avg_preds = np.mean(all_model_preds, axis=0)

    return [
        (f"{filename}_{end_sec}", avg_preds[i])
        for i, end_sec in enumerate(end_secs)
    ]

In [ ]:
def run_inference(models: list, test_dir: Path = CFG.TEST_SOUNDSCAPES_DIR) -> list:
    """
    Iterate over all test soundscapes and collect predictions.
    Returns list of (row_id, pred_array) for every 5-s chunk.
    """
    test_files = sorted(Path(test_dir).glob("*.ogg"))

    if len(test_files) == 0:
        print(
            f"⚠  No .ogg files found in {test_dir}\n"
            "   (This is expected when running locally — "
            "test files are injected at submission time.)"
        )
        return []

    print(f"\nFound {len(test_files)} test soundscapes — running inference…")
    all_results = []

    for i, fp in enumerate(test_files, 1):
        results = predict_soundscape(str(fp), models)
        all_results.extend(results)
        if i % 50 == 0 or i == len(test_files):
            print(f"  [{i}/{len(test_files)}]  {fp.name}  ({len(all_results)} chunks so far)")

    print(f"\nTotal prediction rows : {len(all_results)}")
    return all_results

# 12. BUILD SUBMISSION

In [ ]:
def build_submission(results: list) -> pd.DataFrame:
    """
    Align model predictions with the sample submission row order
    and write submission.csv.

    Any row_id in the sample submission that has no prediction is
    filled with zeros (this can happen if the overlap setting
    produces slightly fewer chunks than expected).
    """
    sample_sub = pd.read_csv(CFG.SAMPLE_SUB_CSV)

    if not results:
        print("No predictions found — returning zeroed sample submission.")
        return sample_sub

    # Dict for fast lookup
    pred_dict = {row_id: preds for row_id, preds in results}

    # Fill in order
    sub_rows = []
    missing  = 0
    for row_id in sample_sub["row_id"]:
        if row_id in pred_dict:
            sub_rows.append(pred_dict[row_id])
        else:
            sub_rows.append(np.zeros(NUM_CLASSES, dtype=np.float32))
            missing += 1

    if missing > 0:
        print(f"⚠  {missing} row_ids had no prediction (filled with 0).")

    preds_array = np.stack(sub_rows, axis=0)
    sub_df = pd.DataFrame(preds_array, columns=ALL_SPECIES)
    sub_df.insert(0, "row_id", sample_sub["row_id"])

    return sub_df

In [ ]:
# ── Run inference & save ────────────────────────────────────────
results    = run_inference(trained_models)
submission = build_submission(results)
submission.to_csv("submission.csv", index=False)

print(f"\n✓ submission.csv saved")
print(f"  Shape  : {submission.shape}")
print(f"  Columns: row_id + {NUM_CLASSES} species")
print(submission.head(3).to_string())
print("\nPrediction stats (non-zero rows):")
pred_vals = submission.iloc[:, 1:].values
print(f"  Mean   : {pred_vals.mean():.4f}")
print(f"  Max    : {pred_vals.max():.4f}")
print(f"  >0.5   : {(pred_vals > 0.5).sum()} cells")